In [ ]:

!pip install azure-storage-file-share

import pandas as pd
from azure.storage.fileshare import ShareDirectoryClient
import io
import numpy as np
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Inventory") \
    .config("spark.driver.maxResultSize", "16g") \
    .getOrCreate()

class FileShareConn:
    '''
    This class gets all the files in Azure's FileShare

    Attributes:
    azure_connection_string (str): Connection string for the FileShare
    file_share_name (int): Name of the FileShare
    dir_path (str): Directory path: Select Oracle, to use .csv files

    Returns:
    list: all the files that are to be transformed
    '''
    def __init__(self, azure_connection_string, file_share_name, dir_path):
        self.azure_connection_string = azure_connection_string
        self.file_share_name = file_share_name
        self.dir_path = dir_path

        self.share_directory_clients = [ShareDirectoryClient.from_connection_string
        (conn_str=azure_connection_string, share_name=file_share_name, directory_path=path) for path in dir_path]
        
    def list_files_recursively(self, directory_client):
        files = []
        for item in directory_client.list_directories_and_files():
            if item.is_directory:
                subdir_client = directory_client.get_subdirectory_client(item.name)
                files.extend(self.list_files_recursively(subdir_client))
            else:
                files.append(directory_client.get_file_client(item.name))
        return files

    def get_all_files(self):
        all_files = []

        for dir_client in self.share_directory_clients:
            file_clients = self.list_files_recursively(dir_client)
            files = [file_client for file_client in file_clients if file_client.get_file_properties().content_settings.content_type == "application/octet-stream"]
            all_files.extend(files)
        return all_files


class sqlTableExists:
    '''
    This class makes a connection with the existing table in the ElectronicPartLog 

    Attributes:
    jdbcHostNameElectronicPartLog (str): Host name of the database
    jdbcPortElectronicPartLog (int): Port number for the database
    jdbcDatabaseElectronicPartLog (str): Name of the database
    jdbcUsernameElectronicPartLog (str): username to access the database
    jdbcPasswordElectronicPartLog (str): password to access the database

    Returns:
    str: Last value of the FilePathName Column
    '''
    def __init__(self, jdbcHostNameElectronicPartLog, jdbcPortElectronicPartLog, jdbcDatabaseElectronicPartLog, jdbcUsernameElectronicPartLog, jdbcPasswordElectronicPartLog):
        self.jdbcHostNameElectronicPartLog = jdbcHostNameElectronicPartLog
        self.jdbcPortElectronicPartLog = jdbcPortElectronicPartLog
        self.jdbcDatabaseElectronicPartLog = jdbcDatabaseElectronicPartLog
        self.jdbcUsernameElectronicPartLog = jdbcUsernameElectronicPartLog
        self.jdbcPasswordElectronicPartLog = jdbcPasswordElectronicPartLog

        self.jdbcUrlElectronicPartLog = f'jdbc:sqlserver://{self.jdbcHostNameElectronicPartLog}:{self.jdbcPortElectronicPartLog};databaseName={self.jdbcDatabaseElectronicPartLog};user={self.jdbcUsernameElectronicPartLog};password={self.jdbcPasswordElectronicPartLog}'
        
    def db_conn(self):

        # Getting the Oracle.Inventory table
        df_SQL_Inventory = spark.read.format('jdbc').option('url', self.jdbcUrlElectronicPartLog).option('dbtable', 'Oracle.Inventory').load()

        df_SQL_Inventory = df_SQL_Inventory.select(
        'InitData',
        'InventId', 
        'UserNumber',
        'ERP',
        'TimeStamp',
        'FilePathName',
        'SrNumber',
        'InventPlace',
        'Quantity',
        'CurrentSituation'
        )

        df_SQL_Inventory = df_SQL_Inventory.orderBy(F.col('TimeStamp').asc(), F.col('FilePathName').asc())
        df_SQL_Inventory = df_SQL_Inventory.withColumn('temp_path', F.substring(df_SQL_Inventory.FilePathName, 1, 11))

        try:
            if df_SQL_Inventory is not None and isinstance(df_SQL_Inventory,DataFrame):

                txt_list = df_SQL_Inventory.groupBy('FilePathName').agg(F.last('FilePathName').alias('lastFilePathName')).rdd.map(lambda x: x.lastFilePathName).collect()
                res = [el.split(', ') for el in txt_list]
                txt = [item.split('/') for inner_list in res for item in inner_list]

        except NameError as error:
            txt = None

        return txt


class sqlTable:
    '''
    The class creates a SQL table, with writing acess in database ElectronicPartLog

    Attributes:
    server_name (str): Host name of the database
    port (int): Port number for the database
    database (str): Name of the database
    username (str): username to access the database
    password (str): password to access the database
    tableName (str): New desired table name to save in the database

    Returns:
    SQL Table in ElectronicPartLog database
    '''
    def __init__(self, server_name, port, database, username, password, tableName):
        self.server_name = server_name
        self.port = port
        self.database = database
        self.username = username
        self.password = password
        self.tableName = tableName
        
    def file_save(self, df):
        '''
        Attributes:
        df (DataFrame): DataFrame to be saved

        Returns:
        Appended SQL table in ElectronicPartLog
        '''

        sqlTableUrl = "jdbc:sqlserver://{}:{};databaseName={}".format(self.server_name, self.port, self.database)

        connection_properties = {
            "user": self.username, 
            "password": self.password, 
            "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
            }

        df.write.mode("append").jdbc(url=sqlTableUrl, table=self.tableName, properties=connection_properties)
        
        return print('{} is done'.format(self.tableName))


class processing_data:
    '''
    This class marks the checkpoint from which the incremental update is to be made and if required transforms the data

    Attributes:
    cnxn (str): Connection string for the FileShare
    increment_path (int): Last path of the FileShare file
    save_sql_server_name (str):name of the sql server to be saved with
    save_sql_port (int): port number of the sql server connection
    save_sql_database (str): name of the database
    save_sql_username (str): username 
    save_sql_password (str): password
    save_sql_table_name (str): new table to be saved 

    Returns:
    DataFrame: Manupulated dataframe 
    '''
    def __init__(self, cnxn, increment_path, save_sql_server_name, save_sql_port, save_sql_database, save_sql_username, save_sql_password, save_sql_table_name):
        self.cnxn = cnxn
        self.increment_path = increment_path   
        self.save_sql_server_name = save_sql_server_name
        self.save_sql_port = save_sql_port
        self.save_sql_database = save_sql_database
        self.save_sql_username = save_sql_username
        self.save_sql_password = save_sql_password
        self.save_sql_table_name = save_sql_table_name

    def fileshare_to_transform(self, black_list):
        files = self.cnxn.get_all_files()
        
        file_names_clients = []

        if files is not None:
            for file_client in files[:]:
                if file_client is not None and file_client.file_path is not None:
                    file_name = file_client.file_path
                    file_names_clients.append((file_name, file_client))

        file_names_clients = [item for item in file_names_clients if item[0] not in black_list]

        files = [file_client for (file_name, file_client) in file_names_clients if file_name not in self.increment_path]

        print(files)
        return files
      
        
    def transform_data(self, black_list_file_names):

        self.black_list_file_names = black_list_file_names

        files = self.fileshare_to_transform(self.black_list_file_names)

        if not files:
            raise Exception('Already updated')

        else: 
        
            for file_client in files[:]: 
        
                file_name = file_client.file_path
                print(file_name)
                file_stream = file_client.download_file().readall()
                file_stream = io.BytesIO(file_stream)

                if 'SAP' in file_name:
                    storage_location, system_type, file_path = file_name[2], file_name[4], '/'.join(file_name)
                    df = pd.read_csv(file_stream, sep='\t', engine='python', header=0)
                    col_name = ''.join(map(str, [x for x in df.columns]))
                    year, month, day = col_name[5:9], col_name[9:11], col_name[11:13]
                    df = df.rename(columns={col_name: 'InitData'})

                    df['FilePathName'] = file_path
        
                    df['InventPlace'] = storage_location

                else:
                    system_type, year, month, day, file_path = file_name[4], file_name[5], file_name[6], file_name[7], '/'.join(file_name)                   
                    df = pd.read_csv(file_stream, sep='\t', engine='python', header=0, names=['InitData'])                    
                    df['FilePathName'] = file_path
                    
                    if '.csv.csv' not in file_path:
                        df['InventPlace'] = df['FilePathName'].str[-7:-4]
                    else:
                        df['InventPlace'] = df['FilePathName'].str[-11:-8]

                df['ERP'] = system_type 
                df[['InventId', 'UserNumber']] = df['InitData'].str.split(r'\s+', n=1, expand=True)
                df['TimeStamp'] = year + month + day
                df['TimeStamp'] = pd.to_datetime(df['TimeStamp'], format='%Y%m%d').dt.date
                df['SrNumber'] = df['InventId'].str[2:9]

                df['Quantity'] = df['InventId'].str[9:22]
                df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce').astype("Int32")

                df['CurrentSituation'] = df['InventId'].str[22:]
                df['CurrentSituation'] = pd.to_numeric(df['CurrentSituation'], errors='coerce').astype("Int32")

                df = spark.createDataFrame(df)
                df = df.select('InitData', 'InventId', 'UserNumber', 'ERP', 'TimeStamp', 'FilePathName', 'SrNumber', 'InventPlace', 'Quantity', 'CurrentSituation')

                sql_table = sqlTable(self.save_sql_server_name, 
                                    self.save_sql_port, 
                                    self.save_sql_database, 
                                    self.save_sql_username, 
                                    self.save_sql_password, 
                                    self.save_sql_table_name)
        
                # Save the DataFrame to the SQL table
                sql_table.file_save(df)

        return print('Your SQL table "[Oracle].[Inventory]" is ready')


if __name__ == '__main__':

    # Create an instance of FileShareConn
    FileShareConnInst = FileShareConn(
        azure_connection_string = "<your_azure_connection_string>",
        file_share_name = "InvIntEnv",
        dir_path = ['IntEnv/InvGen2/V2/PROCESSED/Oracle/', 'IntEnv/InvGen2/E01/PROCESSED/SAP']
        )

    # Create an instance of sqlTableExists
    sqlTableExistsInst= sqlTableExists(
        jdbcHostNameElectronicPartLog='<your_jdbc_host_name>',
        jdbcPortElectronicPartLog='<your_jdbc_port_numner>',
        jdbcDatabaseElectronicPartLog='<your_jdbc_database_name>',
        jdbcUsernameElectronicPartLog='<your_jdbc_user_name>',
        jdbcPasswordElectronicPartLog='<your_jdbc_password>'
        )

    txt_increment_path = sqlTableExistsInst.db_conn()

    # connection details of sqlTable
    server_name = "<your_jdbc_host_name>"
    port = "<your_jdbc_port_numner>"
    database = "<your_jdbc_database_name>"
    username = "<your_jdbc_user_name>"
    password = "<your_jdbc_password>"
    tableName = "<your_jdbc_tablename>"

    # If you have some blacklist files else just use None
    black_list_files = pd.read_parquet('<your_blacklist_data_path>')
                    
    # Create an instance of processing_data
    processingDataInst = processing_data(FileShareConnInst, txt_increment_path, server_name, port, database, username, password, tableName)

    # create an object of processing_data
    processed_data = processingDataInst.transform_data(black_list_files)

